# Architecture hotspots

Find busy containers, deep ownership paths, and dominant semantic types.

This sample uses only standard SysML v2 concepts and automatically discovers the project's `model/` or `src/` directory.

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import syside

def find_sysml_root(start=Path.cwd()):
    """Find the nearest model/ or src/ folder containing textual SysML."""
    for directory in (start, *start.parents):
        for folder_name in ('model', 'src'):
            candidate = directory / folder_name
            if candidate.is_dir() and next(candidate.rglob('*.sysml'), None):
                return candidate
    raise FileNotFoundError('No model/ or src/ directory containing .sysml files was found')

SYSML_ROOT = find_sysml_root()
SYSML_FILES = sorted(SYSML_ROOT.rglob('*.sysml'))
model, diagnostics = syside.try_load_model([str(path) for path in SYSML_FILES])
print(f'Loaded {len(SYSML_FILES)} SysML files from {SYSML_ROOT}')

In [ ]:
from IPython.display import display

semantic_elements = (
    list(model.elements(syside.Usage, include_subtypes=True))
    + list(model.elements(syside.Definition, include_subtypes=True))
)

def safe_name(element):
    value = element.qualified_name or element.name or element.declared_name
    return str(value) if value else '<unnamed>'

def ownership_depth(element):
    depth, current, seen = 0, element.owner, set()
    while current is not None and id(current) not in seen:
        seen.add(id(current))
        depth += 1
        current = getattr(current, 'owner', None)
    return depth

containers = []
for element in semantic_elements:
    children = [child for child in element.owned_elements if isinstance(child, (syside.Usage, syside.Definition))]
    if children:
        containers.append((len(children), type(element).__name__, safe_name(element)))

print('Top containers by directly owned semantic elements')
display(sorted(containers, reverse=True)[:25])

print('Deepest ownership paths')
display(sorted(
    ((ownership_depth(item), type(item).__name__, safe_name(item)) for item in semantic_elements),
    reverse=True,
)[:25])

print('Most common semantic types')
Counter(type(item).__name__ for item in semantic_elements).most_common(30)